In [1]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
from numpy.ma.core import less_equal
from tqdm import tqdm

In [2]:
# Create Model
m = gp.Model("Model_1")

Set parameter Username
Set parameter LicenseID to value 2841584
Academic license - for non-commercial use only - expires 2027-07-06


In [3]:
# GENERAL PARAMETERS

# Number of vehicle types
V = 6

# Number of nodes i,j
nodes = 4

# I, J = nodes, nodes
arcs = {i: [j for j in range(nodes) if abs(i - j) <= 1] for i in range(nodes)}

# Time Steps 12 (days) #testing with +1 day
T = 12

# Commodity variable types
X = [GRB.INTEGER, GRB.CONTINUOUS, GRB.CONTINUOUS, GRB.CONTINUOUS, GRB.CONTINUOUS, GRB.INTEGER]
# Crew, consumables, equipment, samples, propellant, crew return

Y = GRB.INTEGER
# Spacecrafts of same type

In [4]:
# ASSUMPTIONS

# Consumption rates [kg/crew/day]
food_consumption = 1.015
water_consumption = 6.37
oxygen_consumption = 1.18
consumption = food_consumption + water_consumption + oxygen_consumption

# Crew mass [kg/crew]
crew_mass = 100

# Gravitational acceleration [m/sˆ2]
g_0 = 9.8


In [5]:
# VEHICLE DATA

# Structure mass [kg]
s = np.array([38415, 12014, 4841, 6053, 2770, 1719])

# Specific impulses [s]
I_sp = np.array([421, 421, 0, 314, 311, 311])

# Payload Capacity [kg]
C = np.array([0, 0, 524, 60, 500, 250])

# Propellant Capacity [kg]
M = np.array([452045, 107725, 0, 18413, 8804, 2358])


In [6]:
# DISPLACEMENT DATA

# Velocity change [km/s]
# PAC, LEO, LLO, LS are 0, 1, 2, 3
delta_V = {0: {0: 0, 1: 0}, 1: {0: 0, 1: 0, 2: 4.04}, 2: {1: 4.04, 2: 0, 3: 1.87}, 3: {2: 1.87, 3: 0}}

# Time of fare [days]
TOF = {0: {0: 1, 1: 1}, 1: {0: 1, 1: 1, 2: 3}, 2: {1: 3, 2: 1, 3: 1}, 3: {2: 1, 3: 1}}

# Propellant mass fraction
phi = [[{j: 0 if delta_V[i][j] == 0 else
        (1 - np.exp(-(delta_V[i][j] * 1000 / (I_sp[v] * g_0))) if I_sp[v] != 0 else 1) for j in arcs[i]}
        for i in arcs]
       for v in range(V)]


In [7]:
# CREATE COMMODITY FLOW VECTORS AND S/C COMMODITY FLOW

def create_commodity_flow(model, V, arcs, T, X, direction):
    x_flow = [[{j: [np.array([[model.addVar(vtype=X[x], name=f'commodity_{direction}flow_{v},{i},{j},{t},{x}')]
                              for x in range(len(X))])
                    for t in range(T)]
                for j in arcs[i]}
               for i in arcs]
              for v in range(V)]

    return x_flow


def create_sc_commodity_flow(model, V, arcs, T, Y, direction):
    y_flow = [
        [{j: [np.array([model.addVar(vtype=Y, name=f'sc_commodity_{direction}flow_{v},{i},{j},{t}')]) for t in range(T)]
          for j in arcs[i]}
         for i in arcs]
        for v in range(V)]

    return y_flow

# Outflow+ leaving from node i to j, inflow- arriving at node j from i

x_outflow, x_inflow = create_commodity_flow(m, V, arcs, T, X, "out"), create_commodity_flow(m, V, arcs, T, X, "in")
y_outflow, y_inflow = create_sc_commodity_flow(m, V, arcs, T, Y, "out"), create_sc_commodity_flow(m, V, arcs, T, Y, "in")

m.update()


In [8]:
# # Create commodity flow variables and add them to model
# def create_add_commodity_flow(model, V=V, I=I, J=J, T=T):
#     x_outflow, x_inflow = {}, {}
#     for v, i, j, t in tqdm([(v, i, j, t) for v in range(V) for i in range(I) for j in range(J) for t in range(T)]):
#         x_outflow[0, v, i, j, t] = model.addVar(vtype=GRB.INTEGER, name='commodity_outflow_crew')  # Different names??
#         x_inflow[0, v, i, j, t] = model.addVar(vtype=GRB.INTEGER, name='commodity_inflow_crew')
#     for v, i, j, t in tqdm([(v, i, j, t) for v in range(V) for i in range(I) for j in range(J) for t in range(T)]):
#         x_outflow[1, v, i, j, t] = model.addVar(vtype=GRB.CONTINUOUS, name='commodity_outflow_equipment')
#         x_inflow[1, v, i, j, t] = model.addVar(vtype=GRB.CONTINUOUS, name='commodity_inflow_equipment')
#     for v, i, j, t in tqdm([(v, i, j, t) for v in range(V) for i in range(I) for j in range(J) for t in range(T)]):
#         x_outflow[2, v, i, j, t] = model.addVar(vtype=GRB.CONTINUOUS, name='commodity_outflow_samples')
#         x_inflow[2, v, i, j, t] = model.addVar(vtype=GRB.CONTINUOUS, name='commodity_inflow_samples')
#     for v, i, j, t in tqdm([(v, i, j, t) for v in range(V) for i in range(I) for j in range(J) for t in range(T)]):
#         x_outflow[3, v, i, j, t] = model.addVar(vtype=GRB.CONTINUOUS, name='commodity_outflow_consumables')
#         x_inflow[3, v, i, j, t] = model.addVar(vtype=GRB.CONTINUOUS, name='commodity_inflow_consumables')
#     for v, i, j, t in tqdm([(v, i, j, t) for v in range(V) for i in range(I) for j in range(J) for t in range(T)]):
#         x_outflow[4, v, i, j, t] = model.addVar(vtype=GRB.CONTINUOUS, name='commodity_outflow_propellant')
#         x_inflow[4, v, i, j, t] = model.addVar(vtype=GRB.CONTINUOUS, name='commodity_inflow_propellant')
#     return x_outflow, x_inflow
#
#
# x_outflow, x_inflow = create_add_commodity_flow(model=m)
#
#
# # Create number of spacecraft flow variables and add them to model
# def create_add_number_spacecraft_per_arc(model, V=V, I=I, J=J, T=T):
#     y_outflow, y_inflow = {}, {}
#     for v, i, j, t in tqdm([(v, i, j, t) for v in range(V) for i in range(I) for j in range(J) for t in range(T)]):
#         y_outflow[v, i, j, t] = model.addVar(vtype=GRB.INTEGER, name=f"spacecraft_ouflow_{v}{i}{j}{t}")
#         y_inflow[v, i, j, t] = model.addVar(vtype=GRB.INTEGER, name=f"spacecraft_inflow_{v}{i}{j}{t}")
#     return y_outflow, y_inflow
#
#
# y_outflow, y_inflow = create_add_number_spacecraft_per_arc(model=m)
#
# m.update()  #??

In [9]:
# CONSTRAINTS 2 & 3 MASS BALANCE
# Node commodity demand D vectors (positive for supply)
# sum(x[i][t]+) - sum(x[i][t]-) <= D[i][t]
# x = Crew, consumables, equipment, samples, propellant

# COMMODITY DEMAND
# Earth (PAC) crew, consumables, equipment, propellant supply, AND Moon surface sample supply (infinite)
# THE COMMODITIES ARE PROVIDED AT LEO ????
# CHECK DAYS FOR MISSION !!!
D = [[np.array([float('inf') if ((x != 3 and x != 5 and i == 0) or (x == 3 and i == 3)) else 0 for x in range(len(X))])
      for _ in range(T)]
    for i in arcs]

# APOLLO

# for t in range(T):
#     D[1][t] = np.array([999999 if x != 3 else 0 for x in range(len(X))])
#     D[3][t][3] = 999999

# Crew demand/supply
D[3][5][0] = -2 # Lunar surface day 5 crew demand (negative supply)
D[2][4][0] = -1 # Lunar orbit day 4 crew demand
D[3][6][5] = 2 # Lunar surface day 6 crew supply (return)
D[2][7][5] = 1 # Lunar orbit day 7 crew supply (return)
D[0][11][5] = -3 # Earth day 11 crew demand (return)

D[3][5][2] = -420 # Lunar surface day 5 (scientific) equipment demand

D[0][11][3] = -110 # Earth day 11 lunar sample demand



#attempting to move all demand+supply by 1 day
#D[3][5][0] = -2 # Lunar surface day 6 crew demand (negative supply)
#D[2][4][0] = -1 # Lunar orbit day 5 crew demand
#D[3][6][5] = 2 # Lunar surface day 7 crew supply (return)
#D[2][7][5] = 1 # Lunar orbit day 8 crew supply (return)
#D[0][11][5] = -3 # Earth day 12 crew demand (return)

#D[3][5][2] = -420 # Lunar surface day 6 (scientific) equipment demand

#D[0][11][3] = -110 # Earth day 12 lunar sample demand


# S/C COMMODITY DEMAND
d = [[[1 if (i == 0 and t ==0) else 0 for t in range(T)] # Infinite supply of spacecrafts at PAC ???
     for _ in range(V)]
    for i in arcs]


In [10]:
# ADD THE CONSTRAINTS (2 & 3)

for i in arcs:
    for t in range(T):

        x_outflow_sum = sum(x_outflow[v][i][j][t] for v in range(V) for j in arcs[i]) if t < T \
            else np.array([[0] for _ in range(len(X))])
        # On the last day there is no outflow

        x_inflow_sum = sum(x_inflow[v][j][i][t - TOF[j][i]] if t >= TOF[j][i]
                           else np.array([[0] for _ in range(len(X))])
                           for v in range(V) for j in arcs[i])
        # Only count the inflows for which the spacecraft has had time to arrive

        for x in range(len(X)):
            m.addConstr(x_outflow_sum[x][0] - x_inflow_sum[x][0] <= D[i][t][x])

        # S/C commodity supply and demand
        for v in range(V):
            y_outflow_sum = sum(y_outflow[v][i][j][t] for j in arcs[i]) if t < T \
                else np.array([0])

            y_inflow_sum = sum(y_inflow[v][j][i][t - TOF[j][i]] if t >= TOF[j][i]
                               else np.array([0])
                               for j in arcs[i])

            m.addConstr(y_outflow_sum[0] - y_inflow_sum[0] <= d[i][v][t])

m.update()

In [11]:
# CONSTRAINTS 4 COMMODITY TRANSFORMATION

# Commodity transformation matrix
# Q[x+, y+] = [x-, y-] --> Difference between what leaves from node i and what arrives at node j. Ex. propellant use
# x = Crew, consumables, equipment, samples, propellant, crew return

def create_commodity_transformation(V, arcs, consumption, crew_mass, phi, TOF):
    Q = [[{j: np.array([[1, 0, 0, 0, 0, 0, 0],
                        [-consumption * TOF[i][j], 1, 0, 0, 0, -consumption * TOF[i][j], 0], # Consumable consumption
                        [0, 0, 1, 0, 0, 0, 0],
                        [0, 0, 0, 1, 0, 0, 0],
                        [crew_mass * -phi[v][i][j], -phi[v][i][j], -phi[v][i][j], -phi[v][i][j], 1 - phi[v][i][j], crew_mass * -phi[v][i][j], -phi[v][i][j]], # Propellant consumption
                        [0, 0, 0, 0, 0, 1, 0],
                        [0, 0, 0, 0, 0, 0, 1]])
           for j in arcs[i]}
          for i in arcs]
         for v in range(V)]

    return Q

Q = create_commodity_transformation(V, arcs, consumption, crew_mass, phi, TOF)

In [12]:
# ADD THE CONSTRAINTS (4)

for v in range(V):
    for i in arcs:
        for j in arcs[i]:
            for t in range(T-1):
                for leaving, arriving in zip(np.dot(Q[v][i][j], np.concatenate((x_outflow[v][i][j][t],
                                                                                np.array([s[v]*y_outflow[v][i][j][t]])), axis=0)),
                                             np.concatenate((x_inflow[v][i][j][t], np.array([s[v]*y_inflow[v][i][j][t]])), axis=0)):

                    m.addConstr(leaving[0] == arriving[0])


# m.addConstr(Q[v][i][j] * np.concatenate((x_outflow[v][i][j][t], np.array([s[v]*y_outflow[v][i][j][t]])), axis=0) ==
#                             np.concatenate((x_inflow[v][i][j][t], np.array([s[v]*y_inflow[v][i][j][t]])), axis=0))

m.update()


In [13]:
# CONSTRAINTS 5 CONCURRENCY LIMITS

# Concurrency constraint matrix
# H[x+] <= e * y+ --> Payload mass and fuel in s/c does not exceed maximum capacities
# x = Crew, consumables, equipment, samples, propellant, return crew

# def create_concurrency_constraint(V, arcs, crew_mass): # Different vehicles version
#     H = [[{j: np.array([[crew_mass, 1, 1, 1, 0],
#                         [0, 0, 0, 0, 1]])
#            for j in arcs[i]}
#           for i in arcs]
#          for v in range(V)]
#
#     return H


def create_concurrency_constraint(arcs, crew_mass): # Same for all vehicles
    H = [{j: np.array([[crew_mass, 1, 1, 1, 0, crew_mass],
                        [0, 0, 0, 0, 1, 0]])
           for j in arcs[i]}
          for i in arcs]
    return H


def create_sc_design_parameters(V, C, M):
    e = [np.array([[C[v]], [M[v]]]) for v in range(V)]
    return e


H = create_concurrency_constraint(arcs, crew_mass)
e = create_sc_design_parameters(V, C, M)

In [14]:
# ADD THE CONSTRAINTS (5)

for v in range(V):
    for i in arcs:
        for j in arcs[i]:
            for t in range(T-1):
                for commodity, constraint in zip(np.dot(H[i][j], x_outflow[v][i][j][t]),
                                                 e[v]*y_outflow[v][i][j][t][0]):

                    m.addConstr(commodity[0] <= constraint[0])

m.update()


In [15]:
# CONSTRAINTS 6 TIME-WINDOW
# ADD THE CONSTRAINTS (6)

for v in range(V):
    for i in arcs:
        for j in arcs[i]:
            for t in range(T-1):
                for commodity_out in x_outflow[v][i][j][t]:
                    m.addConstr(commodity_out[0] >= 0)

                for commodity_in in x_inflow[v][i][j][t]:
                    m.addConstr(commodity_in[0] >= 0)

                m.addConstr(y_outflow[v][i][j][t][0] >= 0)
                m.addConstr(y_inflow[v][i][j][t][0] >= 0)

m.update()

# s[v] >= 0


In [16]:
# CONSTRAINTS 7 SPACE-CRAFT MASS

In [17]:
# CONSTRAINT 1 COST FUNCTION - INITIAL MASS AT LEO
# sum(cost * x + cost_y * s * y)

# x = Crew, consumables, equipment, samples, propellant, crew(return)


def create_commodity_cost(V, arcs, crew_mass):
    cost_coeff = [[{j:
                        [np.array([[crew_mass], [1], [1], [1], [1],[crew_mass]]) if (i == 1 and j == 2)
                         else np.array([[0] for _ in range(len(X))])
                         for t in range(T-1)]
           for j in arcs[i]}
          for i in arcs]
         for _ in range(V)]

    sc_cost_coeff = [[{j:
                        [1 if (i == 1 and j == 2)
                         else 0
                         for t in range(T-1)]
           for j in arcs[i]}
          for i in arcs]
         for _ in range(V)]

    return cost_coeff, sc_cost_coeff


"""
def create_commodity_cost(V, arcs, crew_mass):
    cost_coeff = [[{j:
                        [np.array([[crew_mass], [1], [1], [1], [1],[crew_mass]]) 
                         for t in range(T-1)]
           for j in arcs[i]}
          for i in arcs]
         for _ in range(V)]

    sc_cost_coeff = [[{j:
                        [1 
                         for t in range(T-1)]
           for j in arcs[i]}
          for i in arcs]
         for _ in range(V)]

    return cost_coeff, sc_cost_coeff
"""
cost_coeff, sc_cost_coeff = create_commodity_cost(V, arcs, crew_mass)

In [18]:
# DEFINE THE COST FUNCTION (1)

cost = sum(
    np.dot(cost_coeff[v][i][j][t].T, x_outflow[v][i][j][t]) + sc_cost_coeff[v][i][j][t] * s[v] * y_outflow[v][i][j][t][0]
    for v in range(V)
    for i in arcs
    for j in arcs[i]
    for t in range(T-1)
)


cost = cost[0][0]

m.setObjective(cost, GRB.MINIMIZE)
m.update()

In [19]:
m.optimize()

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.5.0 25F71)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 15756 rows, 10080 columns and 35782 nonzeros (Min)
Model fingerprint: 0x1452dec2
Model has 462 linear objective coefficients
Variable types: 5760 continuous, 4320 integer (0 binary)
Coefficient statistics:
  Matrix range     [3e-01, 5e+05]
  Objective range  [1e+00, 4e+04]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+100]
         Consider reformulating model or setting NumericFocus parameter
         to avoid numerical issues.

Presolve removed 14580 rows and 8364 columns
Presolve time: 0.00s

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.01 work units)
Thread count was 1 (of 8 available processors)

Solution count 0

Model is infeasible
Best objective -, best bound -, gap -


In [22]:
m.computeIIS()
m.write("infeasible_subsystem.ilp")

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.5.0 25F71)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads


Computing Irreducible Inconsistent Subsystem (IIS)...

           Constraints          |            Bounds           |  Runtime
      Min       Max     Guess   |   Min       Max     Guess   |
--------------------------------------------------------------------------
        0     15696         -         0     10080         -           0s
      288       288       288       165       165       165           1s

IIS computed: 288 constraints, 165 bounds
IIS runtime: 1.15 seconds (0.64 work units)


In [23]:
for c in m.getConstrs():
        if c.IISConstr:  # True if constraint is part of the IIS
            print(f"- Constraint: {c.ConstrName}")


- Constraint: R9
- Constraint: R10
- Constraint: R144
- Constraint: R146
- Constraint: R153
- Constraint: R154
- Constraint: R165
- Constraint: R166
- Constraint: R288
- Constraint: R290
- Constraint: R300
- Constraint: R302
- Constraint: R312
- Constraint: R314
- Constraint: R324
- Constraint: R326
- Constraint: R336
- Constraint: R338
- Constraint: R432
- Constraint: R434
- Constraint: R444
- Constraint: R446
- Constraint: R456
- Constraint: R458
- Constraint: R468
- Constraint: R470
- Constraint: R480
- Constraint: R482
- Constraint: R492
- Constraint: R494
- Constraint: R884
- Constraint: R886
- Constraint: R891
- Constraint: R892
- Constraint: R893
- Constraint: R1038
- Constraint: R1040
- Constraint: R1045
- Constraint: R1047
- Constraint: R1052
- Constraint: R1054
- Constraint: R1059
- Constraint: R1061
- Constraint: R1115
- Constraint: R1117
- Constraint: R1122
- Constraint: R1124
- Constraint: R1129
- Constraint: R1131
- Constraint: R1136
- Constraint: R1138
- Constraint: R114

In [22]:
# x = Crew, consumables, equipment, samples, propellant

results = {final_variable.VarName: final_variable.X for final_variable in m.getVars()}

sorted_results = dict(sorted(results.items(), key=lambda item: int(item[0].split(",")[3])))

f = open("apollo_solution.txt", "w")

# commodity_{direction}flow_{v},{i},{j},{t},{x}

for final in sorted_results:
    if sorted_results[final] != 0:
        print('%s %g' % (final, sorted_results[final]))
        f.write('%s %g' % (final, sorted_results[final]))
        f.write('\n')

f.close()

AttributeError: Unable to retrieve attribute 'X'

In [25]:
import re
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

# ==============================================================================
# 1. NAME MAPPINGS & CONSTANTS
# ==============================================================================
NODE_NAMES = {
    0: "Earth (PAC)",
    1: "Low Earth Orbit (LEO)",
    2: "Low Lunar Orbit (LLO)",
    3: "Lunar Surface (LS)"
}

VEHICLE_NAMES = {
    0: "Saturn V S-II (2nd Stage)",
    1: "Saturn V S-IVB (3rd Stage)",
    2: "Apollo Command Module (CM)",
    3: "Apollo Service Module (SM)",
    4: "LM Descent Stage (LMDS)",
    5: "LM Ascent Stage (LMAS)"
}

COMMODITY_NAMES = {
    0: "Crew (outbound)",
    1: "Consumables (kg)",
    2: "Equipment (kg)",
    3: "Samples (kg)",
    4: "Propellant (kg)",
    5: "Crew Return (inbound)"
}

# Transit times in days for each arc (i, j)
TOF = {
    (0, 0): 1, (0, 1): 1, (1, 0): 1, (1, 1): 1,
    (1, 2): 3, (2, 1): 3, (2, 2): 1, (2, 3): 1,
    (3, 2): 1, (3, 3): 1
}

# ==============================================================================
# 2. PARSER FUNCTION
# ==============================================================================
def parse_solution(file_or_text):
    """Parses raw Gurobi variable strings into clean DataFrames."""
    if "\n" in file_or_text and not file_or_text.endswith(".txt"):
        lines = file_or_text.strip().split("\n")
    else:
        with open(file_or_text, "r") as f:
            lines = f.readlines()

    sc_records = []
    comm_records = []

    # Regex patterns
    sc_pattern = re.compile(r"sc_commodity_(outflow|inflow)_(\d+),(\d+),(\d+),(\d+)\s+([\d.]+)")
    comm_pattern = re.compile(r"commodity_(outflow|inflow)_(\d+),(\d+),(\d+),(\d+),(\d+)\s+([\d.]+)")

    for line in lines:
        line = line.strip()
        if not line:
            continue

        sc_match = sc_pattern.match(line)
        if sc_match:
            direction, v, i, j, t, val = sc_match.groups()
            sc_records.append({
                "direction": direction,
                "v": int(v),
                "i": int(i),
                "j": int(j),
                "t": int(t),
                "value": float(val)
            })
            continue

        comm_match = comm_pattern.match(line)
        if comm_match:
            direction, v, i, j, t, x, val = comm_match.groups()
            comm_records.append({
                "direction": direction,
                "v": int(v),
                "i": int(i),
                "j": int(j),
                "t": int(t),
                "x": int(x),
                "value": float(val)
            })

    sc_df = pd.DataFrame(sc_records)
    comm_df = pd.DataFrame(comm_records)
    return sc_df, comm_df


# ==============================================================================
# 3. MISSION TIMELINE TABLE GENERATOR
# ==============================================================================
def build_mission_summary_table(sc_df, comm_df):
    """Combines spacecraft and cargo into a comprehensive leg-by-leg table."""
    # Filter for active outflow operations
    active_sc = sc_df[sc_df["direction"] == "outflow"].copy()

    legs = []
    for _, sc in active_sc.iterrows():
        v, i, j, t, count = sc["v"], sc["i"], sc["j"], sc["t"], sc["value"]
        t_arr = t + TOF.get((i, j), 1)

        # Get commodities on this specific leg
        out_commodities = comm_df[(comm_df["direction"] == "outflow") &
                                  (comm_df["v"] == v) & (comm_df["i"] == i) &
                                  (comm_df["j"] == j) & (comm_df["t"] == t)]

        in_commodities = comm_df[(comm_df["direction"] == "inflow") &
                                 (comm_df["v"] == v) & (comm_df["i"] == i) &
                                 (comm_df["j"] == j) & (comm_df["t"] == t)]

        # Extract specific items
        cargo_dict = {COMMODITY_NAMES[x_idx]: 0.0 for x_idx in COMMODITY_NAMES}
        for _, c in out_commodities.iterrows():
            cargo_dict[COMMODITY_NAMES[c["x"]]] = c["value"]

        # Calculate fuel burned
        prop_out = out_commodities[out_commodities["x"] == 4]["value"].sum()
        prop_in = in_commodities[in_commodities["x"] == 4]["value"].sum()
        fuel_burned = max(0.0, prop_out - prop_in)

        is_holdover = (i == j)
        leg_type = "Holdover / Stationkeeping" if is_holdover else "Transportation Burn"

        legs.append({
            "Day Depart": t,
            "Day Arrive": t_arr,
            "Origin": NODE_NAMES[i],
            "Destination": NODE_NAMES[j],
            "Vehicle": VEHICLE_NAMES[v],
            "Ships Active": int(count),
            "Operation": leg_type,
            "Crew Outbound": int(cargo_dict["Crew (outbound)"]),
            "Crew Return": int(cargo_dict["Crew Return (inbound)"]),
            "Equipment (kg)": cargo_dict["Equipment (kg)"],
            "Consumables (kg)": round(cargo_dict["Consumables (kg)"], 2),
            "Samples (kg)": cargo_dict["Samples (kg)"],
            "Propellant Carried (kg)": round(prop_out, 2),
            "Fuel Burned (kg)": round(fuel_burned, 2)
        })

    summary_df = pd.DataFrame(legs).sort_values(by=["Day Depart", "Origin", "Vehicle"]).reset_index(drop=True)
    return summary_df


# ==============================================================================
# 4. INTERACTIVE TIME-SPACE TRAJECTORY PLOT (FIGURE 9 REPLICA)
# ==============================================================================
def plot_space_time_diagram(summary_df):
    """Creates an interactive Plotly diagram of the spacecraft paths over time."""
    fig = go.Figure()

    # Node positions on Y-axis
    y_coords = {
        "Earth (PAC)": 0,
        "Low Earth Orbit (LEO)": 1,
        "Low Lunar Orbit (LLO)": 2,
        "Lunar Surface (LS)": 3
    }

    # Distinct colors for each vehicle type
    color_map = {
        "Saturn V S-II (2nd Stage)": "#636EFA",
        "Saturn V S-IVB (3rd Stage)": "#EF553B",
        "Apollo Command Module (CM)": "#00CC96",
        "Apollo Service Module (SM)": "#AB63FA",
        "LM Descent Stage (LMDS)": "#FFA15A",
        "LM Ascent Stage (LMAS)": "#19D3F3"
    }

    vehicles_present = summary_df["Vehicle"].unique()

    # Add small Y-offsets so parallel flights don't overlap completely
    v_offsets = {v: (idx - (len(vehicles_present)-1)/2) * 0.05 for idx, v in enumerate(vehicles_present)}

    used_legend = set()

    for _, row in summary_df.iterrows():
        v = row["Vehicle"]
        y_from = y_coords[row["Origin"]] + v_offsets[v]
        y_to = y_coords[row["Destination"]] + v_offsets[v]
        x_from = row["Day Depart"]
        x_to = row["Day Arrive"]

        is_holdover = (row["Origin"] == row["Destination"])
        line_style = "dot" if is_holdover else "solid"
        line_width = 2 if is_holdover else 4

        # Tooltip text
        tooltip = (
            f"<b>{v}</b> (Count: {row['Ships Active']})<br>"
            f"<b>Route:</b> {row['Origin']} → {row['Destination']}<br>"
            f"<b>Time:</b> Day {x_from} → Day {x_to}<br>"
            f"<b>Crew:</b> {row['Crew Outbound'] + row['Crew Return']} astronauts<br>"
            f"<b>Equipment:</b> {row['Equipment (kg)']} kg<br>"
            f"<b>Samples:</b> {row['Samples (kg)']} kg<br>"
            f"<b>Consumables:</b> {row['Consumables (kg)']} kg<br>"
            f"<b>Propellant Outflow:</b> {row['Propellant Carried (kg)']} kg<br>"
            f"<b>Fuel Burned:</b> {row['Fuel Burned (kg)']} kg"
        )

        show_legend = v not in used_legend
        used_legend.add(v)

        fig.add_trace(go.Scatter(
            x=[x_from, x_to],
            y=[y_from, y_to],
            mode="lines+markers",
            name=v,
            legendgroup=v,
            showlegend=show_legend,
            line=dict(width=line_width, dash=line_style, color=color_map.get(v, "#333")),
            marker=dict(size=7),
            hovertext=tooltip,
            hoverinfo="text"
        ))

    # Layout styling matching paper
    fig.update_layout(
        title="<b>Space Logistics Optimal Trajectory (Apollo 17 Simulation)</b>",
        xaxis=dict(
            title="<b>Mission Timeline (Days)</b>",
            dtick=1,
            range=[-0.5, summary_df["Day Arrive"].max() + 0.5],
            gridcolor="#E1E5ED"
        ),
        yaxis=dict(
            title="<b>Orbital / Surface Nodes</b>",
            tickmode="array",
            tickvals=[0, 1, 2, 3],
            ticktext=["Earth (PAC)", "Low Earth Orbit (LEO)", "Low Lunar Orbit (LLO)", "Lunar Surface (LS)"],
            gridcolor="#E1E5ED"
        ),
        template="plotly_white",
        height=650,
        hovermode="closest",
        legend=dict(title="Spacecraft Fleet", orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )

    return fig


# ==============================================================================
# 5. MAIN EXECUTION BLOCK
# ==============================================================================
if __name__ == "__main__":
    # Point this to your solution file or pass the raw string
    filename = "apollo_solution.txt"

    try:
        sc_df, comm_df = parse_solution(filename)
        summary_table = build_mission_summary_table(sc_df, comm_df)

        print("\n" + "="*80)
        print("                  OPTIMIZED MISSION FLIGHT PLAN SUMMARY")
        print("="*80)

        # Display clean filtered columns for the console
        cols_to_print = ["Day Depart", "Day Arrive", "Origin", "Destination", "Vehicle",
                         "Ships Active", "Crew Outbound", "Crew Return", "Equipment (kg)",
                         "Samples (kg)", "Fuel Burned (kg)"]
        print(summary_table[cols_to_print].to_string(index=False))

        # Export table to CSV
        summary_table.to_csv("apollo_mission_flow_summary.csv", index=False)
        print("\n[+] Full table exported to 'apollo_mission_flow_summary.csv'")

        # Generate and show interactive plot
        fig = plot_space_time_diagram(summary_table)
        fig.show()
        fig.write_html("apollo_mission_spacetime_diagram.html")
        print("[+] Interactive diagram saved to 'apollo_mission_spacetime_diagram.html'")

    except FileNotFoundError:
        print(f"Error: Could not find '{filename}'. Ensure the solution text file is in the same directory.")


                  OPTIMIZED MISSION FLIGHT PLAN SUMMARY
 Day Depart  Day Arrive                Origin           Destination                    Vehicle  Ships Active  Crew Outbound  Crew Return  Equipment (kg)  Samples (kg)  Fuel Burned (kg)
          0           1           Earth (PAC)           Earth (PAC) Apollo Command Module (CM)             1              0            0             0.0           0.0              0.00
          0           1           Earth (PAC) Low Earth Orbit (LEO) Apollo Command Module (CM)             2              3            0           420.0           0.0              0.00
          0           1           Earth (PAC)           Earth (PAC)     LM Ascent Stage (LMAS)             2              0            0             0.0           0.0              0.00
          0           1           Earth (PAC) Low Earth Orbit (LEO)    LM Descent Stage (LMDS)             3              0            0             0.0           0.0              0.00
          0       

[+] Interactive diagram saved to 'apollo_mission_spacetime_diagram.html'


In [23]:
# # EQUATION 7 CONSTRAINTS
#
# # Structural Fraction (fuel dependent)
# alpha = 0.045  # LOX/kerosene
#
# # Gravitational Acceleration Earth
# g_0 = 9.8  # m/s2
#
# # Upper Bound Allowed for Propellant Tank Capacity
# M_ub = 500000  # kg
#
# # Spacecraft Impulsive Burn
# t_b = 120  # s
#
#
# # Structure Mass Variable
# def create_s_star_variables(model, v=V):
#     variables = {}
#     for v in range(V):
#         variables[v] = model.addVar(vtype=GRB.CONTINUOUS, name=f'Structure_Mass_{v}')
#     return variables
#
#
# s_star = create_s_star_variables(model=m)
#
# m.update()

In [24]:
# # CONSTRAINTS 7
#
# for v in tqdm(V):
#     m.addConstr(s_star[v] = 2.3931 * )